In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Make sure to run the cell above to mount your Google Drive before proceeding.

In [7]:
import os
# ── Paths (edit these) ───────────────────────────────────────────────────────
# Option A: dataset next to Lasana under DNN_Project/data/...
# Option B: local images/ and masks/ folders inside Lasana/

BASE_DIR = os.path.abspath(os.getcwd())

# Try common locations in order; first existing pair wins.
_CANDIDATES = [
    (
        os.path.join(BASE_DIR, "dataset", "Forest Segmented", "Forest Segmented", "images"),
        os.path.join(BASE_DIR, "dataset", "Forest Segmented", "Forest Segmented", "masks"),
    ),
    (
        os.path.join(BASE_DIR, "images"),
        os.path.join(BASE_DIR, "masks"),
    ),
    (
        os.path.join(BASE_DIR, "..", "data", "Forest Segmented", "Forest Segmented", "images"),
        os.path.join(BASE_DIR, "..", "data", "Forest Segmented", "Forest Segmented", "masks"),
    ),
]

IMAGE_FOLDER = None
MASK_FOLDER = None
for img_c, mask_c in _CANDIDATES:
    if os.path.isdir(img_c) and os.path.isdir(mask_c):
        IMAGE_FOLDER, MASK_FOLDER = img_c, mask_c
        break

# Manual override if auto-detect fails:
IMAGE_FOLDER = r"/content/drive/MyDrive/DNN-Project/Kalana/images"
MASK_FOLDER  = r"/content/drive/MyDrive/DNN-Project/Kalana/masks"

CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Hyperparameters ──────────────────────────────────────────────────────────
IMG_SIZE = 256
MASK_THRESHOLD = 127
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10   # of full data; test = remainder (~0.10)

BATCH_SIZE = 8     # raise to 16 if GPU memory allows
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
BCE_WEIGHT = 0.5
DICE_WEIGHT = 0.5
EARLY_STOP_PATIENCE = 10
LR_PATIENCE = 5

BEST_CKPT = os.path.join(CHECKPOINT_DIR, "lasana_unet_best.keras")
LAST_CKPT = os.path.join(CHECKPOINT_DIR, "lasana_unet_last.keras")
LOG_CSV = os.path.join(RESULTS_DIR, "training_log.csv")

print("IMAGE_FOLDER:", IMAGE_FOLDER)
print("MASK_FOLDER :", MASK_FOLDER)
assert IMAGE_FOLDER and MASK_FOLDER, (
    "Could not find images/masks. Set IMAGE_FOLDER and MASK_FOLDER manually."
)

IMAGE_FOLDER: /content/drive/MyDrive/DNN-Project/Kalana/images
MASK_FOLDER : /content/drive/MyDrive/DNN-Project/Kalana/masks


# Lasana — Improved Forest Semantic Segmentation

**Task:** Binary forest / non-forest segmentation from satellite images  
**Upgrade from:** `Notebooks/Kalana/main.py` (simple CNN baseline)  
**Model:** U-Net + BatchNorm + skip connections (TensorFlow / Keras)

### What this notebook improves

| Area | Kalana | Lasana |
|------|--------|--------|
| Architecture | Plain encoder–decoder | **U-Net with skip connections** |
| Normalization | None | **BatchNorm** |
| Loss | BCE only | **BCE + Dice** |
| Metrics | Pixel accuracy | **IoU, Dice, Precision, Recall** (+ accuracy) |
| Mask resize | Default interpolation | **Nearest-neighbor** |
| Mask values | `/255` | **Threshold at 127 → 0/1** |
| Augmentation | None | **Flips, rotations, color jitter** |
| Split | 80/20 | **80/10/10 train/val/test** |
| Training | 10 epochs, fixed LR | **Up to 50 epochs, early stop, LR schedule** |

Read `README.md` in this folder for full explanations of **why** each choice is used.

---
### How to run
1. Point `IMAGE_FOLDER` / `MASK_FOLDER` to your dataset (cell below).
2. Runtime with GPU recommended.
3. Run all cells in order.

## 1 · Imports & reproducibility

**Why:** Fixing the random seed makes train/val/test splits and weight init comparable across runs so you can trust that gains come from the model, not luck.

In [8]:
import os
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
    CSVLogger,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2 · Configuration & paths

**Why these settings**
- `IMG_SIZE=256` — matches the Forest Segmented patches and Kalana/Dinura baselines.
- `MASK_THRESHOLD=127` — JPEG masks are rarely pure 0/255; mid-gray pixels must be forced to 0 or 1.
- `80/10/10` split — validation drives early stopping / LR; test stays untouched for final report.
- Longer training + early stopping — U-Net needs more epochs than Kalana’s 10, but we stop when val IoU stalls.

In [9]:
import os
# ── Paths (edit these) ───────────────────────────────────────────────────────
# Option A: dataset next to Lasana under DNN_Project/data/...
# Option B: local images/ and masks/ folders inside Lasana/

BASE_DIR = os.path.abspath(os.getcwd())

# Try common locations in order; first existing pair wins.
_CANDIDATES = [
    (
        os.path.join(BASE_DIR, "dataset", "Forest Segmented", "Forest Segmented", "images"),
        os.path.join(BASE_DIR, "dataset", "Forest Segmented", "Forest Segmented", "masks"),
    ),
    (
        os.path.join(BASE_DIR, "images"),
        os.path.join(BASE_DIR, "masks"),
    ),
    (
        os.path.join(BASE_DIR, "..", "data", "Forest Segmented", "Forest Segmented", "images"),
        os.path.join(BASE_DIR, "..", "data", "Forest Segmented", "Forest Segmented", "masks"),
    ),
]

IMAGE_FOLDER = None
MASK_FOLDER = None
for img_c, mask_c in _CANDIDATES:
    if os.path.isdir(img_c) and os.path.isdir(mask_c):
        IMAGE_FOLDER, MASK_FOLDER = img_c, mask_c
        break

# Manual override if auto-detect fails:
IMAGE_FOLDER = r"/content/drive/MyDrive/DNN-Project/Kalana/images"
MASK_FOLDER  = r"/content/drive/MyDrive/DNN-Project/Kalana/masks"

CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Hyperparameters ──────────────────────────────────────────────────────────
IMG_SIZE = 256
MASK_THRESHOLD = 127
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10   # of full data; test = remainder (~0.10)

BATCH_SIZE = 8     # raise to 16 if GPU memory allows
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
BCE_WEIGHT = 0.5
DICE_WEIGHT = 0.5
EARLY_STOP_PATIENCE = 10
LR_PATIENCE = 5

BEST_CKPT = os.path.join(CHECKPOINT_DIR, "lasana_unet_best.keras")
LAST_CKPT = os.path.join(CHECKPOINT_DIR, "lasana_unet_last.keras")
LOG_CSV = os.path.join(RESULTS_DIR, "training_log.csv")

print("IMAGE_FOLDER:", IMAGE_FOLDER)
print("MASK_FOLDER :", MASK_FOLDER)
assert IMAGE_FOLDER and MASK_FOLDER, (
    "Could not find images/masks. Set IMAGE_FOLDER and MASK_FOLDER manually."
)

IMAGE_FOLDER: /content/drive/MyDrive/DNN-Project/Kalana/images
MASK_FOLDER : /content/drive/MyDrive/DNN-Project/Kalana/masks


## 3 · Load image–mask pairs

**What this does**
- Matches `*_sat_*` images to `*_mask_*` files (same convention as Kalana).
- Resizes **images** with bilinear interpolation (smooth RGB).
- Resizes **masks** with **nearest-neighbor** so class labels stay crisp (no gray edges).
- Binarizes masks with threshold 127.

**Why nearest-neighbor for masks:** bilinear/cubic resize mixes neighboring labels into values like 0.4 — those are not valid class IDs and blur the learning target.

### Optimize data loading: Copying from Google Drive to local storage

To speed up data loading, it's highly recommended to copy the image and mask folders from your mounted Google Drive to the local Colab runtime environment. This significantly reduces I/O latency compared to direct access from Drive.

In [10]:
import shutil

LOCAL_IMAGE_FOLDER = os.path.join(BASE_DIR, "local_images")
LOCAL_MASK_FOLDER = os.path.join(BASE_DIR, "local_masks")

os.makedirs(LOCAL_IMAGE_FOLDER, exist_ok=True)
os.makedirs(LOCAL_MASK_FOLDER, exist_ok=True)

print(f"Copying images from {IMAGE_FOLDER} to {LOCAL_IMAGE_FOLDER}...")
for item in os.listdir(IMAGE_FOLDER):
    s = os.path.join(IMAGE_FOLDER, item)
    d = os.path.join(LOCAL_IMAGE_FOLDER, item)
    if os.path.isfile(s):
        shutil.copy2(s, d)

print(f"Copying masks from {MASK_FOLDER} to {LOCAL_MASK_FOLDER}...")
for item in os.listdir(MASK_FOLDER):
    s = os.path.join(MASK_FOLDER, item)
    d = os.path.join(LOCAL_MASK_FOLDER, item)
    if os.path.isfile(s):
        shutil.copy2(s, d)

# Update the global variables to point to the local copies
IMAGE_FOLDER = LOCAL_IMAGE_FOLDER
MASK_FOLDER = LOCAL_MASK_FOLDER

print("Copy complete. IMAGE_FOLDER and MASK_FOLDER updated to local paths.")
print("New IMAGE_FOLDER:", IMAGE_FOLDER)
print("New MASK_FOLDER :", MASK_FOLDER)

Copying images from /content/drive/MyDrive/DNN-Project/Kalana/images to /content/local_images...
Copying masks from /content/drive/MyDrive/DNN-Project/Kalana/masks to /content/local_masks...
Copy complete. IMAGE_FOLDER and MASK_FOLDER updated to local paths.
New IMAGE_FOLDER: /content/local_images
New MASK_FOLDER : /content/local_masks


In [11]:
# ## 3 · Load image–mask paths (Memory-Optimized)
#
# # IMPORTANT: The previous `load_dataset` function in this cell was loading all images and masks into RAM, causing memory errors. This cell has been replaced.
#
# # The new approach for loading data efficiently is handled by:
# # - **Cell `XvdjTvUcFO8P`**: Defines `load_dataset_paths` to collect only image and mask file paths.
# # - **Cell `PvkiKtYd_PrA`**: Uses `tf.data.Dataset` with `_parse_image_mask_pair` and `make_dataset` to load, preprocess, and augment images and masks on-the-fly in batches, avoiding large memory consumption.
#
# # Please ensure you run cell `XvdjTvUcFO8P` and `PvkiKtYd_PrA` for the new data loading pipeline. Do NOT run any other cells that define or call the old, memory-intensive `load_dataset` function.

# This cell's content is intended to be explanatory markdown text, not executable Python code.
# If you are seeing this, it means the cell was executed as a 'Code' cell.
# Its original purpose was to guide you on the memory-optimized data loading.
# Please ensure you run cells `XvdjTvUcFO8P` and `PvkiKtYd_PrA` for the new data loading pipeline.
# Do NOT run any other cells that define or call the old, memory-intensive `load_dataset` function, as they will cause memory errors.

pass # This 'pass' statement makes the cell valid Python code if accidentally run.

In [12]:
# This cell previously defined and called the 'load_dataset' function, which loaded all images into memory.
# This approach caused '12GB RAM exceeds' errors. This cell has been disabled.
#
# The memory-optimized data loading is now handled by:
# - Cell 'XvdjTvUcFO8P': Defines 'load_dataset_paths' to collect only image and mask file paths.
# - Cell 'PvkiKtYd_PrA': Uses 'tf.data.Dataset' to load, preprocess, and augment data on-the-fly.
#
# Please ensure you run cell 'XvdjTvUcFO8P' and 'PvkiKtYd_PrA' for the new data loading pipeline.
# Do NOT run this cell or any other cells that define or call the old, memory-intensive 'load_dataset' function.

pass # This 'pass' statement makes the cell valid Python code if accidentally run.

In [13]:
# This cell previously defined and called the 'load_dataset' function, which loaded all images into memory.
# This approach caused '12GB RAM exceeds' errors. This cell has been disabled.
#
# The memory-optimized data loading is now handled by:
# - Cell 'XvdjTvUcFO8P': Defines 'load_dataset_paths' to collect only image and mask file paths.
# - Cell 'PvkiKtYd_PrA': Uses 'tf.data.Dataset' to load, preprocess, and augment data on-the-fly.
#
# Please ensure you run cell 'XvdjTvUcFO8P' and 'PvkiKtYd_PrA' for the new data loading pipeline.
# Do NOT run this cell or any other cells that define or call the old, memory-intensive 'load_dataset' function.

pass # This 'pass' statement makes the cell valid Python code if accidentally run.

In [14]:
# This cell previously defined and called the 'load_dataset' function, which loaded all images into memory.
# This approach caused '12GB RAM exceeds' errors. This cell has been disabled.
#
# The memory-optimized data loading is now handled by:
# - Cell 'XvdjTvUcFO8P': Defines 'load_dataset_paths' to collect only image and mask file paths.
# - Cell 'PvkiKtYd_PrA': Uses 'tf.data.Dataset' to load, preprocess, and augment data on-the-fly.
#
# Please ensure you run cell 'XvdjTvUcFO8P' and 'PvkiKtYd_PrA' for the new data loading pipeline.
# Do NOT run this cell or any other cells that define or call the old, memory-intensive 'load_dataset' function.

pass # This 'pass' statement makes the cell valid Python code if accidentally run.

In [15]:
# This cell previously defined and called the 'load_dataset' function, which loaded all images into memory.
# This approach caused '12GB RAM exceeds' errors. This cell has been disabled.
#
# The memory-optimized data loading is now handled by:
# - Cell 'XvdjTvUcFO8P': Defines 'load_dataset_paths' to collect only image and mask file paths.
# - Cell 'PvkiKtYd_PrA': Uses 'tf.data.Dataset' to load, preprocess, and augment data on-the-fly.
#
# Please ensure you run cell 'XvdjTvUcFO8P' and 'PvkiKtYd_PrA' for the new data loading pipeline.
# Do NOT run this cell or any other cells that define or call the old, memory-intensive 'load_dataset' function.

pass # This 'pass' statement makes the cell valid Python code if accidentally run.

In [16]:
import os

# Explicitly defining IMAGE_FOLDER and MASK_FOLDER for this cell's scope
# This assumes the data was successfully copied to local storage in cell 6a6424ec.
# If cell 6a6424ec has not been run, these paths may not contain the data.
IMAGE_FOLDER = r"/content/local_images"
MASK_FOLDER  = r"/content/local_masks"

def find_mask_path(mask_folder, filename):
    """Return best matching mask path for an image filename."""
    stem, ext = os.path.splitext(filename)
    common_exts = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]
    candidate_stems = [stem]
    if "_sat_" in stem:
        candidate_stems.append(stem.replace("_sat_", "_mask_"))

    ext_order = [ext] + [e for e in common_exts if e != ext]
    for candidate_stem in candidate_stems:
        for candidate_ext in ext_order:
            candidate = os.path.join(mask_folder, candidate_stem + candidate_ext)
            if os.path.exists(candidate):
                return candidate
    return None


def load_dataset_paths(image_folder, mask_folder):
    """Returns lists of image and mask paths."""
    image_files = sorted(
        f for f in os.listdir(image_folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"))
    )

    image_paths, mask_paths = [], []
    skipped = 0

    for file in image_files:
        image_path = os.path.join(image_folder, file)
        mask_path = find_mask_path(mask_folder, file)

        # Check if both image and mask exist
        if not os.path.exists(image_path):
            skipped += 1
            continue
        if not mask_path or not os.path.exists(mask_path):
            skipped += 1
            continue

        image_paths.append(image_path)
        mask_paths.append(mask_path)

    if len(image_paths) == 0:
        raise RuntimeError("No valid image-mask pairs found. Check paths.")

    print(f"Found {len(image_paths)} image-mask pairs | skipped {skipped}")
    return image_paths, mask_paths


# Call the new function to get paths
image_paths, mask_paths = load_dataset_paths(IMAGE_FOLDER, MASK_FOLDER)

Found 4649 image-mask pairs | skipped 0


## 4 · Train / Val / Test split

**Why 80/10/10 instead of Kalana’s 80/20**
- **Train** — learn weights.
- **Val** — pick best checkpoint & tune LR / early stop (never used for final claim).
- **Test** — report once at the end so you do not overfit to the number you publish.

In [17]:
# The 'images' and 'masks' arrays, and temporary variables 'X_temp', 'y_temp' are no longer created
# because the data loading has been optimized to use tf.data.Dataset from file paths.
# This cell's original purpose was to split these in-memory arrays.
# The splitting of paths is now handled in cell PvkiKtYd_PrA.

pass # This 'pass' statement makes the cell valid Python code if accidentally run.

## 5 · Data augmentation (train only)

**What:** random horizontal/vertical flips, 90° rotations, mild brightness/contrast changes.  
**Why:** satellite scenes appear at many orientations and lighting conditions; augmentation reduces overfitting without new labels.  
**Important:** val/test are **not** augmented — metrics must reflect real data.

In [18]:
from sklearn.model_selection import train_test_split
import tensorflow as tf # Ensure tf is imported if not already in this scope

# Re-defining key hyperparameters for this cell's scope, to ensure they are available
# These values should match those set in configuration cells like Axl88aiX_Pq9 and bB_FLT6k_Pq-
SEED = 42
IMG_SIZE = 256
MASK_THRESHOLD = 127
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
BATCH_SIZE = 8

def augment_pair(image, mask):
    """Apply the same geometric transform to image and mask."""
    # Horizontal flip
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)

    # Vertical flip
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    # 0 / 90 / 180 / 270 rotation (k * 90°)
    k = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)

    # Photometric jitter on image only (mask labels must stay fixed)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    image = tf.clip_by_value(image, 0.0, 1.0)

    # Keep mask binary after geometric ops
    mask = tf.cast(mask > 0.5, tf.float32)
    return image, mask


def _parse_image_mask_pair(image_path, mask_path, img_size, mask_threshold):
    """Loads, preprocesses, and augments a single image-mask pair."""
    # Load image
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, (img_size, img_size), method=tf.image.ResizeMethod.BILINEAR)

    # Load mask
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_image(mask, channels=1, expand_animations=False)
    mask = tf.image.convert_image_dtype(mask, tf.float32)
    # CRITICAL: nearest-neighbor keeps hard 0/1 labels
    mask = tf.image.resize(mask, (img_size, img_size), method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    mask = tf.cast(mask > (mask_threshold / 255.0), tf.float32)

    return image, mask


def make_dataset(image_paths, mask_paths, batch_size, img_size, mask_threshold, shuffle=False, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))

    # Map the parsing function to load and preprocess data
    ds = ds.map(lambda img_p, mask_p: _parse_image_mask_pair(img_p, mask_p, img_size, mask_threshold), num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(image_paths), 1024), seed=SEED)
    if augment:
        ds = ds.map(augment_pair, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


# Split paths into train/val/test sets
X_train_paths, X_test_paths, y_train_paths, y_test_paths = train_test_split(
    image_paths, mask_paths, test_size=0.10, random_state=SEED
)
val_fraction_of_temp = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
X_train_paths, X_val_paths, y_train_paths, y_val_paths = train_test_split(
    X_train_paths, y_train_paths, test_size=val_fraction_of_temp, random_state=SEED
)

print(f"Train paths: {len(X_train_paths)} | Val paths: {len(X_val_paths)} | Test paths: {len(X_test_paths)}")

train_ds = make_dataset(X_train_paths, y_train_paths, BATCH_SIZE, IMG_SIZE, MASK_THRESHOLD, shuffle=True, augment=True)
val_ds = make_dataset(X_val_paths, y_val_paths, BATCH_SIZE, IMG_SIZE, MASK_THRESHOLD, shuffle=False, augment=False)
test_ds = make_dataset(X_test_paths, y_test_paths, BATCH_SIZE, IMG_SIZE, MASK_THRESHOLD, shuffle=False, augment=False)

print("Datasets ready.")

# The original images, masks, X_temp, y_temp are no longer needed as we are using paths and tf.data
# You might want to remove this line if 'images', 'masks', 'X_temp', 'y_temp' are used later
# del images, masks, X_temp, y_temp

Train paths: 3719 | Val paths: 465 | Test paths: 465
Datasets ready.


## 6 · U-Net model

**What each part does**
- **DoubleConv (Conv → BN → ReLU × 2):** extracts features; BatchNorm stabilizes activations.
- **Encoder + MaxPool:** grows channels, shrinks spatial size (context).
- **Bottleneck:** deepest abstract representation.
- **Decoder + UpSampling:** rebuilds resolution.
- **Skip connections (`concatenate`):** inject encoder detail into decoder so forest edges stay sharp.
- **1×1 Conv + sigmoid:** one probability per pixel (forest vs background).

**Why U-Net beats Kalana’s plain CNN:** without skips, fine boundaries are lost in the bottleneck and the decoder cannot recover them well.

In [19]:
def conv_block(x, filters):
    """Two 3x3 convolutions with BatchNorm + ReLU."""
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x


def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3), features=(64, 128, 256, 512)):
    inputs = Input(shape=input_shape)
    x = inputs
    skips = []

    # Encoder
    for f in features:
        x = conv_block(x, f)
        skips.append(x)
        x = layers.MaxPooling2D(2)(x)

    # Bottleneck
    x = conv_block(x, features[-1] * 2)

    # Decoder
    for f, skip in zip(reversed(features), reversed(skips)):
        x = layers.UpSampling2D(2)(x)
        x = layers.Concatenate()([skip, x])
        x = conv_block(x, f)

    outputs = layers.Conv2D(1, 1, activation="sigmoid", padding="same")(x)
    return Model(inputs, outputs, name="lasana_unet")


model = build_unet()
model.summary()

Model: "lasana_unet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │      1,728 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256, 256,  │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │     36,864 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │     73,728 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │    147,456 │ activation_2[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        512 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ activation_3[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │    294,912 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │      1,024 │ conv2d_4[0][0]  

 Total params: 31,396,609 (119.77 MB)

 Trainable params: 31,384,833 (119.72 MB)

 Non-trainable params: 11,776 (46.00 KB)

## 7 · Loss & metrics

**BCE** — good per-pixel probability calibration.  
**Dice loss** — `1 − Dice`; directly rewards overlap of predicted forest with ground truth.  
**Combined loss** — balances both (same idea as Dinura’s `BCEDiceLoss`).

**IoU / Dice metrics** — what you should report; pixel accuracy alone can look high while forest regions are wrong.

In [20]:
def dice_coef(y_true, y_pred, smooth=1.0):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    bce = tf.reduce_mean(bce)
    return BCE_WEIGHT * bce + DICE_WEIGHT * dice_loss(y_true, y_pred)


def iou_coef(y_true, y_pred, threshold=0.5, smooth=1.0):
    y_pred_bin = tf.cast(y_pred > threshold, tf.float32)
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred_bin, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=bce_dice_loss,
    metrics=[
        "accuracy",
        dice_coef,
        iou_coef,
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)

print("Compiled with BCE + Dice loss; tracking accuracy, Dice, IoU, Precision, Recall.")

Compiled with BCE + Dice loss; tracking accuracy, Dice, IoU, Precision, Recall.


## 8 · Train

**Callbacks**
- **ModelCheckpoint (best val IoU)** — keep the best forest-overlap model, not the last epoch.
- **EarlyStopping** — stop when val IoU stops improving (saves time, fights overfitting).
- **ReduceLROnPlateau** — lower LR when learning stalls.
- **CSVLogger** — save history for plots / reports.

In [ ]:
callbacks = [
    ModelCheckpoint(
        BEST_CKPT,
        monitor="val_iou_coef",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    ModelCheckpoint(
        LAST_CKPT,
        save_best_only=False,
        verbose=0,
    ),
    EarlyStopping(
        monitor="val_iou_coef",
        mode="max",
        patience=EARLY_STOP_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_iou_coef",
        mode="max",
        factor=0.5,
        patience=LR_PATIENCE,
        min_lr=1e-6,
        verbose=1,
    ),
    CSVLogger(LOG_CSV),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=NUM_EPOCHS,
    callbacks=callbacks,
)

print(f"Best checkpoint: {BEST_CKPT}")

Epoch 1/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 0s 666ms/step - accuracy: 0.6894 - dice_coef: 0.6883 - iou_coef: 0.6175 - loss: 0.4692 - precision: 0.7088 - recall: 0.8289
Epoch 1: val_iou_coef improved from None to 0.00033, saving model to /content/checkpoints/lasana_unet_best.keras

Epoch 1: finished saving model to /content/checkpoints/lasana_unet_best.keras
465/465 ━━━━━━━━━━━━━━━━━━━━ 411s 728ms/step - accuracy: 0.7100 - dice_coef: 0.7080 - iou_coef: 0.6428 - loss: 0.4442 - precision: 0.7241 - recall: 0.8523 - val_accuracy: 0.3898 - val_dice_coef: 0.0209 - val_iou_coef: 3.2853e-04 - val_loss: 3.1739 - val_precision: 0.6854 - val_recall: 3.2418e-04 - learning_rate: 0.0010
Epoch 2/50
465/465 ━━━━━━━━━━━━━━━━━━━━ 0s 539ms/step - accuracy: 0.7333 - dice_coef: 0.7242 - iou_coef: 0.6615 - loss: 0.4245 - precision: 0.7387 - recall: 0.8657
Epoch 2: val_iou_coef improved from 0.00033 to 0.65293, saving model to /content/checkpoints/lasana_unet_best.keras

Epoch 2: finished saving model to /content

## 9 · Training curves

**Why plot IoU/Dice, not only loss:** loss can drop while overlap quality stalls. Curves show whether the model is still learning useful forest masks.

In [ ]:
hist = history.history
epochs_range = range(1, len(hist["loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_range, hist["loss"], label="train")
axes[0].plot(epochs_range, hist["val_loss"], label="val")
axes[0].set_title("BCE + Dice Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epochs_range, hist["iou_coef"], label="train")
axes[1].plot(epochs_range, hist["val_iou_coef"], label="val")
axes[1].set_title("IoU")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(epochs_range, hist["dice_coef"], label="train")
axes[2].plot(epochs_range, hist["val_dice_coef"], label="val")
axes[2].set_title("Dice")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()
curve_path = os.path.join(RESULTS_DIR, "training_curves.png")
plt.savefig(curve_path, dpi=150)
plt.show()
print("Saved:", curve_path)

## 10 · Test-set evaluation

Load the **best val-IoU** weights and evaluate once on the held-out test set.  
Compare **IoU / Dice** to Kalana’s pixel accuracy — a fairer view of forest segmentation quality.

In [ ]:
# Reload best weights (EarlyStopping already restored them, but this is explicit)
if os.path.exists(BEST_CKPT):
    model.load_weights(BEST_CKPT)
    print("Loaded best checkpoint.")

results = model.evaluate(test_ds, return_dict=True)
print("\n=== Test metrics ===")
for k, v in results.items():
    print(f"{k:12s}: {v:.4f}")

## 11 · Visual predictions

Side-by-side: original image | ground-truth mask | prediction.  
**What to look for:** sharper forest boundaries than Kalana, fewer missed patches, less “all background” collapse.

In [ ]:
n_show = min(4, len(X_test))
preds = model.predict(X_test[:n_show], verbose=0)
preds_bin = (preds > 0.5).astype(np.float32)

fig, axes = plt.subplots(n_show, 3, figsize=(10, 3 * n_show))
if n_show == 1:
    axes = np.expand_dims(axes, 0)

for i in range(n_show):
    axes[i, 0].imshow(X_test[i])
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(y_test[i].squeeze(), cmap="gray")
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(preds_bin[i].squeeze(), cmap="gray")
    axes[i, 2].set_title("Prediction")
    axes[i, 2].axis("off")

plt.tight_layout()
pred_path = os.path.join(RESULTS_DIR, "prediction_grid.png")
plt.savefig(pred_path, dpi=150)
plt.show()
print("Saved:", pred_path)

## 12 · Summary

You trained an improved baseline that addresses Kalana’s main weaknesses:

1. **U-Net + skips + BatchNorm** → better spatial detail  
2. **BCE + Dice** → optimizes forest overlap, not only pixel-wise BCE  
3. **IoU / Dice metrics** → honest segmentation scores  
4. **Nearest-neighbor masks + threshold 127** → clean labels  
5. **Augmentation + early stopping + LR schedule** → better generalization  

**Artifacts**
- `checkpoints/lasana_unet_best.keras`
- `results/training_log.csv`
- `results/training_curves.png`
- `results/prediction_grid.png`

**Next ideas:** pretrained encoder (ResNet/EfficientNet), Focal loss, threshold tuning on val IoU, or compare against Dinura’s PyTorch U-Net on the same split.